In [1]:
from pyspark.sql.functions import row_number
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    lag, col, lit, radians, sin, 
    cos, asin, sqrt, unix_timestamp, abs, min
)
from pyspark.errors.exceptions.base import PySparkRuntimeError
from pyspark.sql import functions as F
from pyspark.sql.column import Column
from pyspark.sql.window import Window
from dataclasses import dataclass
from pyspark import SparkContext
from datetime import datetime
from pyspark.sql.types import *
import numpy as np
import pandas as pd
import os

In [2]:
@dataclass
class Coordinates:
    latitude: np.float64
    longitude: np.float64
    
    @property
    def point(self) -> tuple:
        return (
            self.latitude,
            self.longitude
        )

In [3]:

# For detecting a moving ship
SOG_MOVE = 1

# 50-nautical-mile
NAUTICAL_MILE = 50

# Collision detecting thresholds. Minimum required time for collision
COLLISION_TIME_WINDOW_MINUTES = 10


# later
EARTH_RADIUS_METERS = 6371000
EARTH_RADIUS_NM = 3440.065


<h3>Big data analytics Task 4</h3>

In [4]:
data_source_folder = "aisdk-2021-12"
paths = [
    os.path.join(
        data_source_folder, 
        data_source_folder + "-" + str(i).rjust(2, "0") + ".csv"
    )
    for i in range(1, 32)
]

print("Using file names")
print(paths[:3])
print("....")

Using file names
['aisdk-2021-12/aisdk-2021-12-01.csv', 'aisdk-2021-12/aisdk-2021-12-02.csv', 'aisdk-2021-12/aisdk-2021-12-03.csv']
....


In [5]:
# Starting new Spark session or getting existing one 
try:
    spark = SparkSession.builder.appName("task_4_cluster").getOrCreate()

    # Test communication with Spark
    spark.range(1).count()

    print(f"Task 4 Spark cluster working. Version = {spark.version}")

except Exception as error:
    print(f"ERROR: {error}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/06 20:07:33 WARN Utils: Your hostname, homedev-25p, resolves to a loopback address: 127.0.1.1; using 192.168.0.104 instead (on interface wlp0s20f3)
26/06/06 20:07:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 20:07:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Task 4 Spark cluster working. Version = 4.1.2


<h3>Data cleaning and pre-processing</h3>

In [6]:
start_time = datetime.now()
start_time_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
print(f"Reading data to Apache Spark start time: {start_time_str}")


data = spark.read.csv(
    paths,
    sep=",",
    header=True,
    inferSchema=True
)

# df_filtered = (
#     data
#     .select("# Timestamp", "MMSI", "Latitude", "Longitude", "Name")
#     .filter(
#         F.col("# Timestamp").isNotNull() &
#         F.col("MMSI").isNotNull() &
#         F.col("Latitude").isNotNull() &
#         F.col("Longitude").isNotNull() &
#         F.col("Name").isNotNull()
#     )
# )
df_filtered = (
    data
    .select("# Timestamp", "MMSI", "Latitude", "Longitude", "Name", "SOG", "Heading")
    .filter(
        F.col("# Timestamp").isNotNull() &
        F.col("MMSI").isNotNull() &
        F.col("Latitude").isNotNull() &
        F.col("Longitude").isNotNull() &
        F.col("Name").isNotNull() &
        F.col("SOG").isNotNull() &
        F.col("Heading").isNotNull()
    )
)

end_time = datetime.now()
end_time_str = end_time.strftime("%Y-%m-%d %H:%M:%S")

time_diff_seconds = (end_time - start_time).seconds

print("----------------------")
print(f"Total {time_diff_seconds} seconds")
print(f"Execution end-time: {end_time_str}")


Reading data to Apache Spark start time: 2026-06-06 20:07:38


----------------------
Total 227 seconds
Execution end-time: 2026-06-06 20:11:26


In [7]:
# Validating if file read is successfull

read_record_count = df_filtered.count()
print(f"Read records {read_record_count} from files .csv")

Read records 245158653 from files .csv


In [8]:
df_filtered = (
    df_filtered
    .withColumn(
        "timestamp",
        F.to_timestamp(
            F.col("# Timestamp"),
            "dd/MM/yyyy HH:mm:ss"
        )
    )
    .select(
        "timestamp",
        "MMSI",
        "Latitude",
        "Longitude",
        "Name",
        "SOG",
        "Heading"
    )
)

In [9]:
window = Window.partitionBy("MMSI").orderBy("timestamp")

In [10]:
df_filtered = df_filtered.withColumn("rn", row_number().over(window))

In [11]:
def haversine_nm(
    a_lat_col: str, 
    a_lon_col: str, 
    b_lat_value: np.float64, 
    b_lon_value: np.float64
):
    return (
        EARTH_RADIUS_NM * 2 * asin(
            sqrt(
                sin((radians(a_lat_col) - radians(lit(b_lat_value))) / 2) ** 2 +
                cos(radians(lit(b_lat_value))) *
                cos(radians(a_lat_col)) *
                sin((radians(a_lon_col) - radians(lit(b_lon_value))) / 2) ** 2
            )
        )
    )
    
def haversine_meters(
    a_lat_col: Column,
    a_lon_col: Column,
    b_lat_col: Column,
    b_lon_col: Column
) -> Column:
    return (
        EARTH_RADIUS_METERS * 2 * asin(
            sqrt(
                sin((radians(a_lat_col) - radians(b_lat_col)) / 2) ** 2 +
                cos(radians(b_lat_col)) *
                cos(radians(a_lat_col)) *
                sin((radians(a_lon_col) - radians(b_lon_col)) / 2) ** 2
            )
        )
    )

In [12]:
COORDINATE_CENTER = Coordinates(
    latitude=55.225000,
    longitude=14.245000
)
print(f"Analyzing ships near: {COORDINATE_CENTER}")

Analyzing ships near: Coordinates(latitude=55.225, longitude=14.245)


In [13]:
# filter out 50 neutilon miles
df_filtered = df_filtered.filter(
    haversine_nm(
        "Latitude", "Longitude",
        COORDINATE_CENTER.latitude, 
        COORDINATE_CENTER.longitude
    ) <= NAUTICAL_MILE
)

In [14]:
ship_records = df_filtered.count()
print(f"Found ships records in radius: {ship_records}")

Found ships records in radius: 23420392


In [15]:
# filter out non stationary vessels
# df_filtered = df_filtered.filter(
#     col("SOG") > SOG_MOVE
# )
# # filter out non stationary vessels
df_filtered = df_filtered.filter(
    col("SOG") > SOG_MOVE
)
df_filtered = (
    df_filtered
    .withColumn("prev_lat", lag("Latitude").over(window))
    .withColumn("prev_lon", lag("Longitude").over(window))
    .withColumn(
        "move_dist_m",
        haversine_meters(
            col("prev_lat"),
            col("prev_lon"),
            col("Latitude"),
            col("Longitude")
        )
    )
)

moving_vessels = (
    df_filtered
    .groupBy("MMSI")
    .agg(
        F.sum(
            F.coalesce(col("move_dist_m"), F.lit(0))
        ).alias("total_move_m")
    )
    .filter(col("total_move_m") > 100)  # vessel moved >100m overall
    .select("MMSI")
)

df_filtered = (
    df_filtered
    .join(moving_vessels, "MMSI", "inner")
    .drop("prev_lat", "prev_lon", "move_dist_m")
)

In [16]:
moving_ships = df_filtered.count()
print(f"Selected moving ships: {moving_ships}")

Selected moving ships: 18371705


<h3>Simple data exploration</h3>

<h3>Collision analysis</h3>

In [17]:
GRID = 0.01  # about 1 km

df_filtered = (
    df_filtered
    .withColumn("lat_bucket", F.floor(F.col("Latitude") / GRID))
    .withColumn("lon_bucket", F.floor(F.col("Longitude") / GRID))
    .withColumn(
        "time_bucket",
        F.floor(
            F.unix_timestamp("timestamp") / 60
        )
    )
)

In [18]:
df_filtered_a = df_filtered.select(
    "timestamp",
    "Name",
    "MMSI",
    "Latitude",
    "Longitude",
    "SOG",
    "Heading",
    "lat_bucket",
    "lon_bucket",
    "time_bucket"
).alias("a")

df_filtered_b = df_filtered.select(
    "timestamp",
    "Name",
    "MMSI",
    "Latitude",
    "Longitude",
    "SOG",
    "Heading",
    "lat_bucket",
    "lon_bucket",
    "time_bucket"
).alias("b")

In [19]:
candidate_pairs = (
    df_filtered_a
    .join(
        df_filtered_b,
        (col("a.MMSI") < col("b.MMSI"))
        &
        (col("a.time_bucket") == col("b.time_bucket"))
        &
        (col("a.lat_bucket") == col("b.lat_bucket"))
        &
        (col("a.lon_bucket") == col("b.lon_bucket"))
    )
    .select(
        col("a.MMSI").alias("mmsi_a"),
        col("a.timestamp").alias("timestamp_a"),
        col("a.Name").alias("name_a"),
        col("a.Latitude").alias("lat_a"),
        col("a.Longitude").alias("lon_a"),
        col("a.SOG").alias("sog_a"),
        col("a.Heading").alias("heading_a"),
        col("b.MMSI").alias("mmsi_b"),
        col("b.timestamp").alias("timestamp_b"),
        col("b.Name").alias("name_b"),
        col("b.Latitude").alias("lat_b"),
        col("b.Longitude").alias("lon_b"),
        col("b.SOG").alias("sog_b"),
        col("b.Heading").alias("heading_b")
    )
)
candidate_pairs = candidate_pairs.withColumn(
    "heading_diff",
    F.least(
        F.abs(F.col("heading_a") - F.col("heading_b")),
        360 - F.abs(F.col("heading_a") - F.col("heading_b"))
    )
)
candidate_pairs = candidate_pairs.filter(F.col("heading_diff") > 30)

In [20]:
dist_df = candidate_pairs.withColumn(
    "distance_m",
    haversine_meters(
        col("lat_a"), col("lon_a"),
        col("lat_b"), col("lon_b")
    )
)

min_dist = dist_df.agg(F.min("distance_m")).first()[0]

closest_pair = (
    dist_df
    .filter(F.col("distance_m") == min_dist)
    .first()
)

In [21]:
closest_pair_dict = closest_pair.asDict()
print(f"Ship pair (Ship A, Ship B), which is close to collision:\n")
for key, value in closest_pair_dict.items():
    print(f"{key} : {value}")

Ship pair (Ship A, Ship B), which is close to collision:

mmsi_a : 219019287
timestamp_a : 2021-12-03 16:02:32
name_a : HG 162 NORTH OCEAN
lat_a : 55.243472
lon_a : 15.087747
sog_a : 5.4
heading_a : 220
mmsi_b : 219021428
timestamp_b : 2021-12-03 16:02:54
name_b : HG 165 SOUTH OCEAN
lat_b : 55.243482
lon_b : 15.087745
sog_b : 1.6
heading_b : 173
heading_diff : 47
distance_m : 1.1191536635713002


In [22]:
path_a = df_filtered.filter(
    (F.col("MMSI") == closest_pair_dict["mmsi_a"])
    &
    (
        F.abs(
            F.unix_timestamp("timestamp")
            - F.unix_timestamp(F.lit(closest_pair_dict["timestamp_a"]))
        ) <= COLLISION_TIME_WINDOW_MINUTES * 60 
    )
)
path_a.show(truncate=False)

+---------+-------------------+---------+---------+------------------+---+-------+-----+----------+----------+-----------+
|MMSI     |timestamp          |Latitude |Longitude|Name              |SOG|Heading|rn   |lat_bucket|lon_bucket|time_bucket|
+---------+-------------------+---------+---------+------------------+---+-------+-----+----------+----------+-----------+
|219019287|2021-12-03 15:52:32|55.248097|15.088783|HG 162 NORTH OCEAN|1.6|157    |21426|5524      |1508      |27308992   |
|219019287|2021-12-03 15:52:35|55.248077|15.088792|HG 162 NORTH OCEAN|1.6|164    |21427|5524      |1508      |27308992   |
|219019287|2021-12-03 15:52:38|55.248062|15.088802|HG 162 NORTH OCEAN|1.9|167    |21428|5524      |1508      |27308992   |
|219019287|2021-12-03 15:52:41|55.248032|15.088803|HG 162 NORTH OCEAN|2.3|169    |21429|5524      |1508      |27308992   |
|219019287|2021-12-03 15:52:45|55.24799 |15.088822|HG 162 NORTH OCEAN|2.7|168    |21430|5524      |1508      |27308992   |
|219019287|2021-

In [23]:
path_b = df_filtered.filter(
    (F.col("MMSI") == closest_pair_dict["mmsi_b"])
    &
    (
        F.abs(
            F.unix_timestamp("timestamp")
            - F.unix_timestamp(F.lit(closest_pair_dict["timestamp_b"]))
        ) <= COLLISION_TIME_WINDOW_MINUTES * 60 
    )
)
path_b.show(10, truncate=False)

+---------+-------------------+---------+---------+------------------+---+-------+-----+----------+----------+-----------+
|MMSI     |timestamp          |Latitude |Longitude|Name              |SOG|Heading|rn   |lat_bucket|lon_bucket|time_bucket|
+---------+-------------------+---------+---------+------------------+---+-------+-----+----------+----------+-----------+
|219021428|2021-12-03 15:52:54|55.24762 |15.088092|HG 165 SOUTH OCEAN|2.4|166    |22624|5524      |1508      |27308992   |
|219021428|2021-12-03 15:52:56|55.247595|15.0881  |HG 165 SOUTH OCEAN|2.4|166    |22625|5524      |1508      |27308992   |
|219021428|2021-12-03 15:53:01|55.247532|15.08814 |HG 165 SOUTH OCEAN|2.4|164    |22626|5524      |1508      |27308993   |
|219021428|2021-12-03 15:53:04|55.247505|15.088155|HG 165 SOUTH OCEAN|2.4|163    |22627|5524      |1508      |27308993   |
|219021428|2021-12-03 15:53:06|55.247468|15.08818 |HG 165 SOUTH OCEAN|2.4|164    |22628|5524      |1508      |27308993   |
|219021428|2021-

<h3>Results export to csv</h3>

In [25]:
path_a.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("ship_a_collision")
    
print("Exported Ship A collision data to ship_a_collision folder")

Exported Ship A collision data to ship_a_collision folder


In [26]:
path_b.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("ship_b_collision")
    
print("Exported Ship B collision data to ship_b_collision folder")

Exported Ship B collision data to ship_b_collision folder


In [27]:
spark.stop()

In [28]:
# The objective of this examination is to 
# evaluate your ability to process large-scale temporal and spatial data. 
# You are required to identify two vessels that have collided 
# (or experienced the closest possible physical proximity indicating a collision) 
# within a specified marine area. You must visualize their respective trajectories 
# 10 minutes prior to and 10 minutes following the time of collision.

In [29]:
# Geographic Area: 
# 1. Filter the dataset to isolate vessels operating 
# within a 50-nautical-mile (nm) radius of a center coordinate 
# located at Latitude: 55.225000, Longitude: 14.245000.

# 2. Vessel State: You are looking specifically for moving vessels 
# that intersect in time and space, resulting in a collision.
# You must implement logic to identify and filter out stationary 
# vessels (e.g., ships at anchor or safely docked adjacent to one another).

# 3. Data Integrity: AIS data frequently contains errors. 
# You must account for and filter out GPS anomalies and data noise. 
# This is critical to ensure that a sudden jump in GPS coordinates 
# is not falsely identified as a collision.
